# Module 7: Optimization Theory

Optimization is the engine that drives machine learning. Every time a model "learns", it is minimizing a loss function or maximizing a utility function. This module covers the theoretical foundations and practical algorithms of mathematical optimization in AI.

## Contents
1. Convex Sets and Convex Functions
2. Unconstrained Optimization & Gradient Descent Variants (SGD, Momentum, Adam)
3. Second-Order Optimization (Newton's Method)
4. Constrained Optimization & Lagrange Multipliers
5. Duality and KKT Conditions
6. Convergence Analysis

## 1. Convex Sets and Convex Functions

A set $C \subseteq \mathbb{R}^n$ is **convex** if the line segment between any two points in $C$ lies entirely within $C$. Formally, for any $x, y \in C$ and $\theta \in [0, 1]$:
$$\theta x + (1 - \theta) y \in C$$

A function $f: \mathbb{R}^n \to \mathbb{R}$ is **convex** if its domain is a convex set and for all $x, y \in \text{dom } f$ and $\theta \in [0, 1]$:
$$f(\theta x + (1 - \theta) y) \le \theta f(x) + (1 - \theta) f(y)$$

### First-Order and Second-Order Conditions
- **First-order condition**: A differentiable function $f$ is convex iff:
  $$f(y) \ge f(x) + \nabla f(x)^T (y - x) \quad \forall x, y$$
  This means the tangent line/hyperplane at any point lies *below* the function.
- **Second-order condition**: A twice-differentiable function $f$ is convex iff its Hessian is positive semi-definite (PSD) everywhere:
  $$\nabla^2 f(x) \succeq 0 \quad \forall x$$

Let's visualize a convex vs. non-convex function in Python.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-2, 2, 100)
y_convex = x**2
y_nonconvex = x**3 - 2*x

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(x, y_convex, 'b-', label='$f(x) = x^2$ (Convex)')
ax[0].fill_between(x, y_convex, 4, color='blue', alpha=0.1)
ax[0].set_title("Convex Function")
ax[0].legend()
ax[0].grid(True)

ax[1].plot(x, y_nonconvex, 'r-', label='$f(x) = x^3 - 2x$ (Non-convex)')
ax[1].set_title("Non-convex Function")
ax[1].legend()
ax[1].grid(True)

plt.show()

## 2. Unconstrained Optimization & Gradient Descent

To minimize an unconstrained differentiable function $f(x)$, we start at an initial point $x_0$ and update iteratively:
$$x_{k+1} = x_k - \alpha_k \nabla f(x_k)$$
where $\alpha_k > 0$ is the learning rate.

### Gradient Descent Variants in AI:
1. **Stochastic Gradient Descent (SGD)**: Estimates the gradient using a single sample or mini-batch.
2. **Momentum**: Adds a fraction of the previous update to the current step to accelerate gradient vectors in the right direction:
   $$v_{k+1} = \beta v_k + (1 - \beta) \nabla f(x_k)$$
   $$x_{k+1} = x_k - \alpha v_{k+1}$$
3. **Adam (Adaptive Moment Estimation)**: Computes adaptive learning rates for each parameter using running averages of both the first ($m_t$) and second ($v_t$) moments of the gradients:
   $$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
   $$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
   $$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
   $$x_{t+1} = x_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

In [ ]:
def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def grad_rosenbrock(x, y):
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])

# Simple Gradient Descent implementation
def gd(grad_fn, init_point, lr=0.001, steps=100):
    path = [init_point]
    curr = np.array(init_point, dtype=float)
    for _ in range(steps):
        curr -= lr * grad_fn(curr[0], curr[1])
        path.append(curr.copy())
    return np.array(path)

path = gd(grad_rosenbrock, [-1.5, 1.5], lr=0.001, steps=2000)
print("Final point after 2000 steps:", path[-1])

## 3. Second-Order Optimization (Newton's Method)

Newton's method uses second-order Taylor expansion to approximate the function locally as a quadratic:
$$f(x + \Delta x) \approx f(x) + \nabla f(x)^T \Delta x + \frac{1}{2} \Delta x^T \nabla^2 f(x) \Delta x$$

Minimizing this quadratic yields the Newton step:
$$\Delta x = - \left( \nabla^2 f(x) \right)^{-1} \nabla f(x)$$

Newton's method converges quadratically (much faster than gradient descent's linear convergence) but requires computing and inverting the $n \times n$ Hessian matrix $\nabla^2 f(x)$, which is computationally prohibitive ($O(n^3)$) for neural networks with millions of parameters.

In [ ]:
def newton_method(grad_fn, hess_fn, init_point, steps=10):
    curr = np.array(init_point, dtype=float)
    path = [curr.copy()]
    for _ in range(steps):
        g = grad_fn(curr[0], curr[1])
        H = hess_fn(curr[0], curr[1])
        step = -np.linalg.solve(H, g)
        curr += step
        path.append(curr.copy())
    return np.array(path)

def hess_rosenbrock(x, y):
    hxx = 2 - 400 * y + 1200 * x**2
    hxy = -400 * x
    hyy = 200
    return np.array([[hxx, hxy], [hxy, hyy]])

path_newton = newton_method(grad_rosenbrock, hess_rosenbrock, [-1.5, 1.5], steps=20)
print("Newton final point after 20 steps:", path_newton[-1])

## 4. Constrained Optimization & Lagrange Multipliers

To solve problems of the form:
$$\min_{x} f(x) \quad \text{subject to } g_i(x) = 0 \quad (i=1,\dots,m)$$

We construct the **Lagrangian**:
$$\mathcal{L}(x, \lambda) = f(x) + \sum_{i=1}^m \lambda_i g_i(x)$$

where $\lambda_i$ are **Lagrange multipliers**. The stationary points of $\mathcal{L}$ give the constrained critical points:
$$\nabla_x \mathcal{L}(x, \lambda) = 0, \quad \nabla_\lambda \mathcal{L}(x, \lambda) = 0$$

In [ ]:
import sympy as sp

# Minimize f(x, y) = x^2 + y^2 subject to g(x, y) = x + y - 1 = 0
x, y, lam = sp.symbols('x y lam')
f = x**2 + y**2
g = x + y - 1
L = f + lam * g

eq1 = sp.diff(L, x)
eq2 = sp.diff(L, y)
eq3 = sp.diff(L, lam)

sol = sp.solve([eq1, eq2, eq3], (x, y, lam))
print("Optimal constrained solution:", sol)

## 5. Duality and KKT Conditions

For inequality constrained optimization:
$$\min_x f(x) \quad \text{s.t. } g_i(x) \le 0 \quad (i=1,\dots,m), \quad h_j(x) = 0 \quad (j=1,\dots,p)$$

The **Karush-Kuhn-Tucker (KKT) Conditions** are first-order necessary conditions for a point to be optimal:

1. **Stationarity**: $\nabla f(x^*) + \sum_{i=1}^m \lambda_i^* \nabla g_i(x^*) + \sum_{j=1}^p \nu_j^* \nabla h_j(x^*) = 0$
2. **Primal Feasibility**: $g_i(x^*) \le 0$ and $h_j(x^*) = 0$
3. **Dual Feasibility**: $\lambda_i^* \ge 0$
4. **Complementary Slackness**: $\lambda_i^* g_i(x^*) = 0$

If the problem is convex, KKT conditions are also **sufficient** for optimality. This is crucial for Support Vector Machines (SVMs).

## 6. Convergence Analysis

An optimization algorithm's speed is analyzed via its convergence rate. 
- **Lipschitz Continuity**: $\|\nabla f(x) - \nabla f(y)\| \le L \|x - y\|$
- **Strong Convexity**: $f(y) \ge f(x) + \nabla f(x)^T(y-x) + \frac{m}{2}\|y-x\|^2$

For $L$-smooth and $m$-strongly convex functions, Gradient Descent with a fixed step size $\alpha \le 1/L$ converges at a linear rate:
$$\|x_k - x^*\|^2 \le \left(1 - \frac{m}{L}\right)^k \|x_0 - x^*\|^2$$

This illustrates the importance of the **condition number** $\kappa = L/m$. When $\kappa$ is large, the optimization problem is ill-conditioned (narrow valleys), making standard gradient descent very slow and necessitating momentum or adaptive algorithms.